## Import Module

In [1]:
import arcpy
import os
import datetime
from arcpy.sa import *

## Projecting DEMs into CRS 32119

In [2]:
# --- 1. Define Paths ---
base_dir = r"C:\Users\<YOUR_USERNAME>\Documents\ArcGIS\Projects\DEM_04232026"
ot_folder = os.path.join(base_dir, "OT_DEM")
noaa_folder = os.path.join(base_dir, "NOAA_DEM")
proj_folder = os.path.join(base_dir, "Projected_Tiles")

if not os.path.exists(proj_folder):
    os.makedirs(proj_folder)

# --- 2. Global Environment Settings ---
arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")

# Set Target CRS: NAD 1983 StatePlane North Carolina FIPS 3200 (Meters)
out_crs = arcpy.SpatialReference(32119)
arcpy.env.outputCoordinateSystem = out_crs

# Enable 100% CPU usage to prevent UI freezing
arcpy.env.parallelProcessingFactor = "100%"

# --- 3. The Resolution Fix ---
# Force outputs to be exactly 10x10 meters
arcpy.env.cellSize = 10

# Ensure Snap Raster is clear before we begin the dynamic projection
arcpy.env.snapRaster = None

print("Cell 1 Complete: Environments locked. Ready to establish dynamic anchor.")

Cell 1 Complete: Environments locked. Ready to establish dynamic anchor.


In [3]:
# Real-time logging function
def log_step(message):
    current_time = datetime.datetime.now().strftime("%H:%M:%S")
    arcpy.AddMessage(f"[{current_time}] {message}") # Pushes to Geoprocessing history
    print(f"[{current_time}] {message}") # Pushes to Notebook output

print("Cell 2 Complete: Logging function ready.")

Cell 2 Complete: Logging function ready.


In [16]:
projected_tiles = []
log_step("--- STARTING PROJECTION PHASE ---")

# --- STEP A: Establish the Dynamic Metric Anchor using OT Data ---
arcpy.env.workspace = ot_folder
ot_rasters = arcpy.ListRasters("*.tif")

if ot_rasters:
    first_raw_ot = ot_rasters[0]
    first_proj_ot_name = f"OT_proj_{first_raw_ot}"
    first_proj_ot_path = os.path.join(proj_folder, first_proj_ot_name)
    
    log_step(f"Establishing Metric Anchor Grid using: {first_raw_ot}...")
    arcpy.management.ProjectRaster(
        in_raster=os.path.join(ot_folder, first_raw_ot), 
        out_raster=first_proj_ot_path, 
        out_coor_system=out_crs, 
        resampling_type="BILINEAR",
        cell_size="10 10"  # force cell size, global env setting failed to work here
    )
    projected_tiles.append(first_proj_ot_path)
    
    # LOCK IN THE SNAP RASTER!
    arcpy.env.snapRaster = first_proj_ot_path
    log_step(f"✅ Master Snap Raster locked to: {first_proj_ot_name}")
    
    # Project the REST of the OT tiles
    for idx, ras in enumerate(ot_rasters[1:], 2):
        in_path = os.path.join(ot_folder, ras)
        out_path = os.path.join(proj_folder, f"OT_proj_{ras}")
        
        log_step(f"  -> [{idx}/{len(ot_rasters)}] Projecting OT tile {ras}...")
        arcpy.management.ProjectRaster(
            in_raster=in_path, out_raster=out_path, 
            out_coor_system=out_crs, resampling_type="BILINEAR",
            cell_size="10 10"  # force cell size, global env setting failed to work here
        )
        projected_tiles.append(out_path)
else:
    log_step("❌ ERROR: No OT tiles found to use as an anchor!")

# --- STEP B: Project NOAA Data (Automatically snapping to the OT grid) ---
arcpy.env.workspace = noaa_folder
noaa_rasters = arcpy.ListRasters("*.tif")

if noaa_rasters:
    for idx, ras in enumerate(noaa_rasters, 1):
        in_path = os.path.join(noaa_folder, ras)
        out_path = os.path.join(proj_folder, f"NOAA_proj_{ras}")
        
        log_step(f"  -> [{idx}/{len(noaa_rasters)}] Projecting NOAA tile {ras} to 10m grid...")
        arcpy.management.ProjectRaster(
            in_raster=in_path, out_raster=out_path, 
            out_coor_system=out_crs, resampling_type="BILINEAR",
            cell_size="10 10"  # force cell size, global env setting failed to work here
        )
        projected_tiles.append(out_path)

log_step(f"Projection complete! {len(projected_tiles)} tiles perfectly aligned.")

[22:43:53] --- STARTING PROJECTION PHASE ---
[22:43:53] Establishing Metric Anchor Grid using: output_USGS10m_1.tif...
[22:45:40] ✅ Master Snap Raster locked to: OT_proj_output_USGS10m_1.tif
[22:45:40]   -> [2/4] Projecting OT tile output_USGS10m_2.tif...
[22:47:24]   -> [3/4] Projecting OT tile output_USGS10m_3.tif...
[22:48:49]   -> [4/4] Projecting OT tile output_USGS10m_4.tif...
[22:49:18]   -> [1/5] Projecting NOAA tile NC_Middle1_GCS_3m_NAVDm.tif to 10m grid...
[22:49:58]   -> [2/5] Projecting NOAA tile NC_Middle2_GCS_3m_NAVDm.tif to 10m grid...
[22:50:28]   -> [3/5] Projecting NOAA tile NC_Northern_GCS_3m_NAVDm.tif to 10m grid...
[22:50:45]   -> [4/5] Projecting NOAA tile NC_Southern2_GCS_3m_NAVDm.tif to 10m grid...
[22:51:30]   -> [5/5] Projecting NOAA tile VA_Southern_GCS_3m_NAVDm.tif to 10m grid...
[22:52:03] Projection complete! 9 tiles perfectly aligned.


## Checking through projected raster tiles

In [25]:
arcpy.env.workspace = proj_folder

proj_rasters = arcpy.ListRasters("*.tif")

if proj_rasters:
    for ras in proj_rasters:
        file_path = os.path.join(proj_folder, ras)
        desc = arcpy.Describe(file_path)     
        sr = desc.spatialReference
        
        print(f"File: {ras}")
        print(f"  X Cell Size: {desc.meanCellWidth}")
        print(f"  Y Cell Size: {desc.meanCellHeight}")

        # --- NEW METADATA PROPERTIES ---
        print(f"  Coordinate System: {sr.name}")
        print(f"  Linear Unit: {sr.linearUnitName}") # E.g., 'Meter' or 'Foot_US'
        
        print(f"  Pixel Type: {desc.pixelType}")     # E.g., 'F32' (32-bit float) or 'U8' (8-bit integer)
        print(f"  Band Count: {desc.bandCount}")
        print(f"  NoData Value: {desc.noDataValue}") 
        # -------------------------------
        
        print("-" * 50)
else:
    log_step("No TIF files found in the Projected Tiles folder.")

File: NOAA_proj_NC_Middle1_GCS_3m_NAVDm.tif
  X Cell Size: 10.0
  Y Cell Size: 10.0
  Coordinate System: NAD_1983_StatePlane_North_Carolina_FIPS_3200
  Linear Unit: Meter
  Pixel Type: F32
  Band Count: 1
  NoData Value: -9999.0
--------------------------------------------------
File: NOAA_proj_NC_Middle2_GCS_3m_NAVDm.tif
  X Cell Size: 10.0
  Y Cell Size: 10.0
  Coordinate System: NAD_1983_StatePlane_North_Carolina_FIPS_3200
  Linear Unit: Meter
  Pixel Type: F32
  Band Count: 1
  NoData Value: -9999.0
--------------------------------------------------
File: NOAA_proj_NC_Northern_GCS_3m_NAVDm.tif
  X Cell Size: 10.0
  Y Cell Size: 10.0
  Coordinate System: NAD_1983_StatePlane_North_Carolina_FIPS_3200
  Linear Unit: Meter
  Pixel Type: F32
  Band Count: 1
  NoData Value: -9999.0
--------------------------------------------------
File: NOAA_proj_NC_Southern2_GCS_3m_NAVDm.tif
  X Cell Size: 10.0
  Y Cell Size: 10.0
  Coordinate System: NAD_1983_StatePlane_North_Carolina_FIPS_3200
  Linea

## Mosaic Tiles

In [26]:
log_step("--- STARTING MOSAIC PHASE ---")

arcpy.env.workspace = base_dir 
mosaic_dem = "Seamless_DEM_32119_10m.tif"

if len(projected_tiles) > 0:
    log_step("Blending tiles together... this may take a few minutes depending on CPU.")
    arcpy.management.MosaicToNewRaster(
        input_rasters=projected_tiles, 
        output_location=base_dir, 
        raster_dataset_name_with_extension=mosaic_dem, 
        coordinate_system_for_the_raster=out_crs, 
        pixel_type="32_BIT_FLOAT", 
        number_of_bands=1, 
        mosaic_method="BLEND" # Smooths the borders where datasets overlap
    )
    log_step(f"✅ Mosaic successfully saved as: {mosaic_dem}")
else:
    log_step("❌ ERROR: No projected tiles found to mosaic.")

[00:46:30] --- STARTING MOSAIC PHASE ---
[00:46:30] Blending tiles together... this may take a few minutes depending on CPU.
[00:49:37] ✅ Mosaic successfully saved as: Seamless_DEM_32119_10m.tif


## Calculate Slope

In [27]:
log_step("--- STARTING SLOPE CALCULATION ---")

mosaic_path = os.path.join(base_dir, mosaic_dem)
slope_output = "Project_Slope_10m.tif"

if arcpy.Exists(mosaic_path):
    from arcpy.sa import *
    
    # Metadata confirmed NAVDm (Meters) and EPSG 5703 (Meters). Map units are Meters.
    # Therefore, Z-Factor MUST be 1.
    log_step("Calculating Planar Slope in Degrees (Z-Factor = 1)...")
    out_slope = Slope(
        in_raster=mosaic_path, 
        output_measurement="DEGREE", 
        method="PLANAR", 
        z_factor=1 
    )
    
    out_slope.save(os.path.join(base_dir, slope_output))
    log_step(f"✅ Slope layer successfully saved as: {slope_output}")
    log_step("--- 🎉 ALL WORKFLOW STEPS COMPLETE 🎉 ---")
else:
    log_step("❌ ERROR: Mosaic file not found. Cannot calculate slope.")

[00:50:45] --- STARTING SLOPE CALCULATION ---
[00:50:45] Calculating Planar Slope in Degrees (Z-Factor = 1)...
[00:53:58] ✅ Slope layer successfully saved as: Project_Slope_10m.tif
[00:53:58] --- 🎉 ALL WORKFLOW STEPS COMPLETE 🎉 ---


## Resample Slope to 30m — Snap to Repository Template Grid

In [9]:
log_step("--- STARTING STEP 4: RESAMPLE & CLIP TO TEMPLATE ---")

snap_raster_path = os.path.join(base_dir, "Reference","apnep-wsi-conserve-wetlands.tif")
slope_10m_path = os.path.join(base_dir, "Project_Slope_10m.tif")
slope_30m_path = os.path.join(base_dir, "Project_Slope_30m.tif")

snap_desc = arcpy.Describe(snap_raster_path)

# 1. Lock the global environments strictly to the template
arcpy.env.snapRaster = snap_raster_path
arcpy.env.cellSize = snap_raster_path
arcpy.env.outputCoordinateSystem = out_crs
arcpy.env.extent = snap_desc.extent
arcpy.env.mask = snap_raster_path # Critical for the extraction step

# 2. Resample (Changes pixel size to 30x30, but leaves the shape too big)
log_step("Resampling to 30m in memory...")
temp_resampled = Resample(
    raster=slope_10m_path,
    resampling_type="Bilinear",
    output_cellsize=30
)

# 3. Extract by Mask (Forces the raster to crop to the 11378 x 9812 template extent!)
log_step("Clipping shape to match WSI template...")
final_slope_30m = ExtractByMask(
    in_raster=temp_resampled,
    in_mask_data=snap_raster_path
)

# 4. Save to disk
final_slope_30m.save(slope_30m_path)

# 5. Clear environments
arcpy.env.snapRaster = None
arcpy.env.cellSize = None
arcpy.env.outputCoordinateSystem = None
arcpy.env.extent = None
arcpy.env.mask = None

# Validation check to print to your console
final_desc = arcpy.Describe(slope_30m_path)
log_step(f"✅ SUCCESS! Final Output Shape: {final_desc.width} x {final_desc.height}")

[14:00:06] --- STARTING STEP 4: RESAMPLE & CLIP TO TEMPLATE ---
[14:00:07] Resampling to 30m in memory...
[14:00:07] Clipping shape to match WSI template...
[14:00:26] ✅ SUCCESS! Final Output Shape: 11378 x 9812


## Slope degrees recoding

> This section of the notebook is drafted after the kernel is restarted after reopen project files so the import are duplicated here

In [1]:
import arcpy
import os
import numpy as np
import datetime
from arcpy.sa import *

def log_step(message):
  current_time = datetime.datetime.now().strftime("%H:%M:%S")
  arcpy.AddMessage(f"[{current_time}] {message}")
  print(f"[{current_time}] {message}")

base_dir = r"C:\Users\<YOUR_USERNAME>\Documents\ArcGIS\Projects\DEM_04232026"
slope_30m_path = os.path.join(base_dir, "Project_Slope_30m.tif")
snap_raster_path = os.path.join(base_dir, "Reference", "apnep-wsi-conserve-wetlands.tif")
coded_slope_path = os.path.join(base_dir, "apnep-wsi-slope.tif")

arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference(32119)
arcpy.env.snapRaster = snap_raster_path
arcpy.env.parallelProcessingFactor = "100%"

for label, path in [("Slope 30m", slope_30m_path), ("Snap raster", snap_raster_path)]:
  if not arcpy.Exists(path):
      raise FileNotFoundError(f"{label} not found: {path}")
  log_step(f"{label}: FOUND -> {path}")

log_step("Configuration complete.")

[17:10:39] Slope 30m: FOUND -> C:\Users\<YOUR_USERNAME>\Documents\ArcGIS\Projects\DEM_04232026\Project_Slope_30m.tif
[17:10:39] Snap raster: FOUND -> C:\Users\<YOUR_USERNAME>\Documents\ArcGIS\Projects\DEM_04232026\Reference\apnep-wsi-conserve-wetlands.tif
[17:10:39] Configuration complete.


In [2]:
log_step("--- SLOPE SUITABILITY RECODE ---")

log_step("Recoding slope degrees to suitability scores...")
slope_raster = Float(Raster(slope_30m_path))

coded_slope = Float(
            Con(slope_raster <= 1, 1.0,
            Con(slope_raster <= 2, 0.75,
            Con(slope_raster <= 3, 0.5,
            Con(slope_raster <= 5, 0.25, 0.0))))
)

coded_slope.save(coded_slope_path)
log_step(f"Slope suitability saved: {coded_slope_path}")

[17:10:42] --- SLOPE SUITABILITY RECODE ---
[17:10:42] Recoding slope degrees to suitability scores...
[17:10:53] Slope suitability saved: C:\Users\<YOUR_USERNAME>\Documents\ArcGIS\Projects\DEM_04232026\apnep-wsi-slope.tif


In [3]:
log_step("Validating slope suitability...")
desc = arcpy.Describe(coded_slope_path)
print(f"File: {coded_slope_path}")
print(f"  Cell Size: {desc.meanCellWidth} x {desc.meanCellHeight}")
print(f"  CRS: {desc.spatialReference.name}")
print(f"  Pixel Type: {desc.pixelType}")
print(f"  Dimensions: {desc.width} x {desc.height}")
print(f"  NoData: {desc.noDataValue}")

slope_arr = arcpy.RasterToNumPyArray(coded_slope_path, nodata_to_value=np.nan)
valid = slope_arr[~np.isnan(slope_arr)]
total = int(valid.size)
labels = {1.0: "0-1 deg", 0.75: "1-2 deg", 0.5: "2-3 deg", 0.25: "3-5 deg", 0.0: ">5 deg"}

print(f"\n{'Suitability':<14}{'Range':<12}{'Pixels':>15}{'%':>8}")
print("-" * 50)
for score in [1.0, 0.75, 0.5, 0.25, 0.0]:
  cnt = int(np.isclose(valid, score).sum())
  pct = 100.0 * cnt / total
  print(f"{score:<14.2f}{labels[score]:<12}{cnt:>15,}{pct:>7.1f}%")
print(f"{'Total':<26}{total:>15,}{'100.0%':>8}")

del slope_arr, valid
log_step("Slope validation complete.")

[17:10:55] Validating slope suitability...
File: C:\Users\<YOUR_USERNAME>\Documents\ArcGIS\Projects\DEM_04232026\apnep-wsi-slope.tif
  Cell Size: 30.0 x 30.0
  CRS: NAD_1983_StatePlane_North_Carolina_FIPS_3200
  Pixel Type: F32
  Dimensions: 11378 x 9812
  NoData: -3.4028234663852886e+38

Suitability   Range                Pixels       %
--------------------------------------------------
1.00          0-1 deg          42,099,374   63.5%
0.75          1-2 deg           8,912,185   13.5%
0.50          2-3 deg           5,326,103    8.0%
0.25          3-5 deg           5,838,013    8.8%
0.00          >5 deg            4,071,360    6.1%
Total                          66,247,035  100.0%
[17:10:58] Slope validation complete.
